<a href="https://colab.research.google.com/github/juliapereira-create/TCC---2026/blob/Mudan%C3%A7a-04%2F11/Mudan%C3%A7a_de_c%C3%B3digo_04_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 1 — Instalação e importação do ambiente
#
# Instalamos:
#   - PuLP 2.8.0: modelagem de PLIM (o "motor" do trabalho)
#   - Pandas: exibição das tabelas de resultado
#
# Não usamos nenhuma biblioteca extra (nada de AHP, heurística
# ou ML). O trabalho é PLIM puro, conforme orientação.
# ─────────────────────────────────────────────────────────────

!pip install pulp==2.8.0 pandas -q

import pulp
import pandas as pd
import numpy as np   # usado apenas para estatísticas descritivas
                     # (média, desvio-padrão) na fase de comparação

print("Ambiente pronto.")
print(f"  PuLP versão: {pulp.__version__}")

Ambiente pronto.
  PuLP versão: 2.8.0


In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 2 — Documentação da origem dos dados (atende item 5)
#
# O orientador pediu para esclarecer se os parâmetros são dados
# REAIS ou HIPÓTESES. Esta célula registra isso formalmente,
# para que a metodologia do TCC fique transparente.
#
# IMPORTANTE: ajuste as strings abaixo conforme a realidade
# do seu estudo. O padrão abaixo assume que as DURAÇÕES vêm
# de referências públicas (ANVISA/fabricante) e os VENCIMENTOS
# seguem a prática do hospital, mas a CAPACIDADE é estimada.
# ─────────────────────────────────────────────────────────────

FONTE_DADOS = {
    'duracao_h': (
        "Estimada a partir de manuais de fabricantes e "
        "referências da ANVISA para equipamentos médico-"
        "hospitalares (RDC 02/2010 e literatura de engenharia "
        "clínica). Valores tratados como PARÂMETROS DE REFERÊNCIA."
    ),
    'mes_vencimento': (
        "Definido pela política interna de manutenção preventiva "
        "do hospital (periodicidade mensal para os equipamentos "
        "do escopo). Tratado como DADO REAL de planejamento."
    ),
    'capacidade_mensal': (
        "ESTIMADA a partir de: (nº técnicos) × (horas úteis/mês) "
        "× (fator de disponibilidade para MP). O fator reflete "
        "que a equipe também atende corretivas, deslocamentos e "
        "atividades administrativas. É uma HIPÓTESE CALIBRÁVEL."
    ),
    'escopo_equipamentos': (
        "Definido pelo setor de engenharia clínica do hospital "
        "como os equipamentos críticos de imagem e cirurgia. "
        "DADO REAL."
    ),
}

print("=" * 70)
print(" ORIGEM DOS DADOS DO MODELO")
print("=" * 70)
for chave, texto in FONTE_DADOS.items():
    print(f"\n▸ {chave}:")
    print(f"    {texto}")

 ORIGEM DOS DADOS DO MODELO

▸ duracao_h:
    Estimada a partir de manuais de fabricantes e referências da ANVISA para equipamentos médico-hospitalares (RDC 02/2010 e literatura de engenharia clínica). Valores tratados como PARÂMETROS DE REFERÊNCIA.

▸ mes_vencimento:
    Definido pela política interna de manutenção preventiva do hospital (periodicidade mensal para os equipamentos do escopo). Tratado como DADO REAL de planejamento.

▸ capacidade_mensal:
    ESTIMADA a partir de: (nº técnicos) × (horas úteis/mês) × (fator de disponibilidade para MP). O fator reflete que a equipe também atende corretivas, deslocamentos e atividades administrativas. É uma HIPÓTESE CALIBRÁVEL.

▸ escopo_equipamentos:
    Definido pelo setor de engenharia clínica do hospital como os equipamentos críticos de imagem e cirurgia. DADO REAL.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 3 — Dados do problema
#
# Mudanças em relação à versão anterior:
#   1) CAPACIDADE_MENSAL_H foi REDUZIDA para um valor
#      operacionalmente plausível (item 2 do feedback).
#      Antes: 3 × 176 = 528 h (100% do tempo só em MP).
#      Agora: 3 × 176 × 0,40 = 211 h (40% do tempo em MP).
#      O fator 0,40 é uma HIPÓTESE CALIBRÁVEL — deixei como
#      parâmetro para você justificar no TCC com dados do
#      hospital (ex.: proporção histórica MP vs. corretiva).
#   2) Cada ordem agora carrega o campo 'periodicidade_meses',
#      usado na restrição R4 (intervalo mínimo entre MPs
#      consecutivas do mesmo equipamento).
# ─────────────────────────────────────────────────────────────

MESES = list(range(1, 13))
NOMES_MESES = ['Jan','Fev','Mar','Abr','Mai','Jun',
               'Jul','Ago','Set','Out','Nov','Dez']

# ── Capacidade operacional efetiva (item 2 do feedback) ──────
NUM_TECNICOS       = 3
HORAS_UTEIS_MES    = 176          # 22 dias × 8 h
FATOR_DISPONIBILIDADE_MP = 0.40   # 40% do tempo em MP
                                  # (resto: corretivas, deslocamento, admin)

CAPACIDADE_MENSAL_H = int(NUM_TECNICOS * HORAS_UTEIS_MES * FATOR_DISPONIBILIDADE_MP)

print("=" * 70)
print(" CAPACIDADE OPERACIONAL")
print("=" * 70)
print(f"  Técnicos:              {NUM_TECNICOS}")
print(f"  Horas úteis/mês:       {HORAS_UTEIS_MES}")
print(f"  Fator disponível p/ MP:{FATOR_DISPONIBILIDADE_MP:.0%}")
print(f"  → Capacidade mensal:   {CAPACIDADE_MENSAL_H} h")
print(f"  → Capacidade anual:    {CAPACIDADE_MENSAL_H * 12} h")

# ── Catálogo de equipamentos ────────────────────────────────
# 'periodicidade_meses' = intervalo mínimo entre duas MPs
# consecutivas do MESMO equipamento (item 3 do feedback).
# Aqui é 1 mês para todos (MP mensal), mas o campo fica
# explícito para que a restrição R4 seja verificável.
EQUIPAMENTOS = [
    # (nome, duracao_h, unidades, periodicidade_meses)
    ('Tomógrafo',        8, 1, 1),
    ('Ressonância',     10, 1, 1),
    ('Mamógrafo',        4, 1, 1),
    ('Raio-X',           4, 1, 1),
    ('Ultrassom',        3, 6, 1),
    ('Arco Cirúrgico',   5, 2, 1),
    ('Hemodinâmica',     8, 2, 1),
]

def gerar_ordens(nome_eq, duracao_h, unidades, periodicidade):
    """
    Gera as ordens de MP de um equipamento.
    - Para cada unidade e cada mês de vencimento, cria UMA ordem.
    - Janela = [mês anterior, mês de vencimento].
    - Guarda 'periodicidade_meses' para a restrição R4.
    """
    ordens = []
    for u in range(1, unidades + 1):
        sufixo = f' U{u}' if unidades > 1 else ''
        for mv in MESES:
            ordens.append({
                'equipamento': nome_eq + sufixo,
                'duracao_h':   duracao_h,
                'mes_min':     max(1, mv - 1),
                'mes_max':     mv,
                'mes_venc':    mv,
                'periodicidade_meses': periodicidade,
            })
    return ordens

TODAS_AS_ORDENS = []
for nome, dur, unid, per in EQUIPAMENTOS:
    TODAS_AS_ORDENS += gerar_ordens(nome, dur, unid, per)

for i, o in enumerate(TODAS_AS_ORDENS):
    o['id'] = i

carga_total = sum(o['duracao_h'] for o in TODAS_AS_ORDENS)

print()
print("=" * 70)
print(" DEMANDA DE MANUTENÇÃO PREVENTIVA")
print("=" * 70)
print(f"  Ordens a programar:   {len(TODAS_AS_ORDENS)}")
print(f"  Carga total:          {carga_total} h")
print(f"  Capacidade anual:     {CAPACIDADE_MENSAL_H * 12} h")
print(f"  Folga anual:          {CAPACIDADE_MENSAL_H*12 - carga_total} h")
print(f"  Carga média mensal:   {carga_total/12:.1f} h")
print(f"  Utilização média:     {carga_total/(CAPACIDADE_MENSAL_H*12):.1%}")

 CAPACIDADE OPERACIONAL
  Técnicos:              3
  Horas úteis/mês:       176
  Fator disponível p/ MP:40%
  → Capacidade mensal:   211 h
  → Capacidade anual:    2532 h

 DEMANDA DE MANUTENÇÃO PREVENTIVA
  Ordens a programar:   168
  Carga total:          840 h
  Capacidade anual:     2532 h
  Folga anual:          1692 h
  Carga média mensal:   70.0 h
  Utilização média:     33.2%


In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 4 — Formulação do modelo PLIM (itens 1, 2 e 3)
#
# ── VARIÁVEIS DE DECISÃO ────────────────────────────────────
#   x[o, m] ∈ {0,1}  → ordem 'o' executada no mês 'm'
#
# ── VARIÁVEIS AUXILIARES (para a FO linear) ─────────────────
#   carga_m     ≥ 0  → carga total alocada no mês 'm'
#   desvio_m    ≥ 0  → |carga_m - carga_media|  (balanceamento)
#   sobrec_m    ≥ 0  → max(0, carga_m - capacidade) (penalidade)
#
# ── FUNÇÃO OBJETIVO (item 1 do feedback) ────────────────────
#   min  w1 · Σ_m desvio_m
#      + w2 · Σ_m sobrec_m
#      + w3 · Σ_o Σ_m (m/max) · x[o,m]      ← antecipação leve
#
#   Interpretação:
#     w1 → espalha a carga uniformemente pelos 12 meses
#     w2 → evita estourar a capacidade (folga operacional)
#     w3 → desempate: prefere início da janela (menor risco)
#   Pesos justificados: w2 > w1 > w3, porque estourar a
#   capacidade é o pior cenário operacional, seguido de
#   concentração de carga. Antecipação é apenas desempate.
#
# ── RESTRIÇÕES ──────────────────────────────────────────────
#   R1. Cada ordem executada EXATAMENTE UMA VEZ no ano.
#   R2. Janela [mes_min, mes_max] (garantida na criação de x).
#   R3. Carga mensal ≤ capacidade efetiva.
#   R4. Intervalo entre MPs consecutivas do MESMO equipamento
#       ≥ periodicidade_meses (item 3 do feedback).
#   R5. Definição linear de desvio_m  (carga_m - média).
#   R6. Definição linear de sobrec_m  (carga_m - capacidade).
# ─────────────────────────────────────────────────────────────

# ── Pesos da função objetivo ────────────────────────────────
W_BALANCEAMENTO = 1.0   # peso do balanceamento
W_SOBRECARGA    = 10.0  # peso da sobrecarga (10× mais caro)
W_ANTECIPACAO   = 0.10  # peso da antecipação (só desempate)

CARGA_MEDIA = carga_total / 12.0

modelo = pulp.LpProblem('Programacao_MP_2026_v2', pulp.LpMinimize)

# ── Variáveis x[o,m] ────────────────────────────────────────
x = {}
for o in TODAS_AS_ORDENS:
    for m in MESES:
        if o['mes_min'] <= m <= o['mes_max']:
            x[(o['id'], m)] = pulp.LpVariable(
                f"x_{o['id']}_m{m}", cat=pulp.LpBinary)

# ── Variáveis auxiliares ────────────────────────────────────
carga_m  = {m: pulp.LpVariable(f"carga_m{m}", lowBound=0) for m in MESES}
desvio_m = {m: pulp.LpVariable(f"desvio_m{m}", lowBound=0) for m in MESES}
sobrec_m = {m: pulp.LpVariable(f"sobrec_m{m}", lowBound=0) for m in MESES}

# ── Função objetivo (multiobjetivo ponderado) ───────────────
modelo += (
    W_BALANCEAMENTO * pulp.lpSum(desvio_m[m] for m in MESES)
    + W_SOBRECARGA   * pulp.lpSum(sobrec_m[m] for m in MESES)
    + W_ANTECIPACAO  * pulp.lpSum(
        (m / 12.0) * x[(o['id'], m)]
        for o in TODAS_AS_ORDENS
        for m in MESES
        if (o['id'], m) in x
    )
), 'FO_multiobjetivo'

# ── R1: cada ordem executada exatamente uma vez ─────────────
for o in TODAS_AS_ORDENS:
    modelo += (
        pulp.lpSum(x[(o['id'], m)] for m in MESES
                   if (o['id'], m) in x) == 1,
        f'R1_unicidade_{o["id"]}'
    )

# ── R3: capacidade mensal ───────────────────────────────────
# Obs.: R2 já está garantida porque x só existe dentro da janela.
for m in MESES:
    modelo += (
        pulp.lpSum(o['duracao_h'] * x[(o['id'], m)]
                   for o in TODAS_AS_ORDENS
                   if (o['id'], m) in x)
        <= CAPACIDADE_MENSAL_H,
        f'R3_capacidade_m{m}'
    )

# ── R4: intervalo mínimo entre MPs consecutivas ─────────────
# Para cada equipamento, ordena as ordens pelo mês de vencimento
# e impõe que a alocação da ordem seguinte seja ≥ a anterior + P.
# Como o vencimento é mensal e P=1, isso é quase automático,
# mas a restrição EXPLICITA o requisito operacional.
from collections import defaultdict
ordens_por_eq = defaultdict(list)
for o in TODAS_AS_ORDENS:
    ordens_por_eq[o['equipamento']].append(o)

for eq, ordens in ordens_por_eq.items():
    ordens_sorted = sorted(ordens, key=lambda o: o['mes_venc'])
    P = ordens_sorted[0]['periodicidade_meses']
    for k in range(len(ordens_sorted) - 1):
        o1 = ordens_sorted[k]
        o2 = ordens_sorted[k + 1]
        # mês alocado da ordem k+1  ≥  mês alocado da ordem k  +  P
        modelo += (
            pulp.lpSum(m * x[(o2['id'], m)] for m in MESES
                       if (o2['id'], m) in x)
            - pulp.lpSum(m * x[(o1['id'], m)] for m in MESES
                         if (o1['id'], m) in x)
            >= P,
            f'R4_intervalo_{eq.replace(" ", "_")}_{k}'
        )

# ── R5: definição linear do desvio em relação à média ───────
for m in MESES:
    modelo += (carga_m[m] == pulp.lpSum(
        o['duracao_h'] * x[(o['id'], m)]
        for o in TODAS_AS_ORDENS if (o['id'], m) in x),
        f'R5a_carga_m{m}')
    modelo += (desvio_m[m] >= carga_m[m] - CARGA_MEDIA, f'R5b_desv_pos_m{m}')
    modelo += (desvio_m[m] >= CARGA_MEDIA - carga_m[m], f'R5c_desv_neg_m{m}')

# ── R6: definição linear da sobrecarga ──────────────────────
for m in MESES:
    modelo += (
        sobrec_m[m] >= carga_m[m] - CAPACIDADE_MENSAL_H,
        f'R6_sobrecarga_m{m}'
    )

# ── Resolução ───────────────────────────────────────────────
solver = pulp.PULP_CBC_CMD(msg=0)
status = modelo.solve(solver)

print("=" * 70)
print(" RESULTADO DO SOLVER")
print("=" * 70)
print(f"  Status:                   {pulp.LpStatus[status]}")
print(f"  Valor da FO (multiobjetivo): {pulp.value(modelo.objective):.2f}")
print(f"  Variáveis de decisão:     {len(x)}")
print(f"  Restrições:               {len(modelo.constraints)}")

 RESULTADO DO SOLVER
  Status:                   Optimal
  Valor da FO (multiobjetivo): 9.10
  Variáveis de decisão:     322
  Restrições:               382


In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 5 — Validação (itens 3 e 4 do feedback)
#
# Verificamos:
#   V1. Todas as ordens foram programadas?
#   V2. Todas dentro da janela?
#   V3. Capacidade mensal respeitada?
#   V4. Intervalo entre MPs consecutivas do mesmo equipamento
#       é ≥ periodicidade? (validação explícita de R4)
# ─────────────────────────────────────────────────────────────

# ── Monta cronograma ────────────────────────────────────────
linhas = []
for o in TODAS_AS_ORDENS:
    for m in MESES:
        if (o['id'], m) in x and pulp.value(x[(o['id'], m)]) > 0.5:
            linhas.append({
                'Equipamento':  o['equipamento'],
                'Duração (h)':  o['duracao_h'],
                'Mês alocado':  NOMES_MESES[m-1],
                'Mês mínimo':   NOMES_MESES[o['mes_min']-1],
                'Mês máximo':   NOMES_MESES[o['mes_max']-1],
                '_mes_num':     m,
                '_mes_venc':    o['mes_venc'],
                '_period':      o['periodicidade_meses'],
            })

df = pd.DataFrame(linhas).sort_values(['_mes_num', 'Equipamento'])
df_view = df.drop(columns=['_mes_num', '_mes_venc', '_period'])

print("=" * 70)
print(" CRONOGRAMA OTIMIZADO — 2026")
print("=" * 70)
print(df_view.to_string(index=False))

# ── V1 ──────────────────────────────────────────────────────
print("\n--- V1: todas as ordens programadas? ---")
assert len(df) == len(TODAS_AS_ORDENS), "❌ Alguma ordem ficou fora!"
print(f"  ✅ {len(df)} / {len(TODAS_AS_ORDENS)} ordens programadas")

# ── V2 ──────────────────────────────────────────────────────
print("\n--- V2: todas dentro da janela? ---")
viol = 0
for _, row in df.iterrows():
    mi = NOMES_MESES.index(row['Mês mínimo']) + 1
    ma = NOMES_MESES.index(row['Mês máximo']) + 1
    al = NOMES_MESES.index(row['Mês alocado']) + 1
    if not (mi <= al <= ma):
        viol += 1
assert viol == 0, f"❌ {viol} MPs fora da janela!"
print(f"  ✅ 0 violações")

# ── V3 ──────────────────────────────────────────────────────
print("\n--- V3: capacidade mensal respeitada? ---")
carga_por_mes = df.groupby('Mês alocado')['Duração (h)'].sum()
carga_por_mes = carga_por_mes.reindex(NOMES_MESES, fill_value=0)
for mes, c in carga_por_mes.items():
    ok = '✅' if c <= CAPACIDADE_MENSAL_H else '❌'
    print(f"  {mes}: {c:>5.0f} h / {CAPACIDADE_MENSAL_H} h  {ok}")
assert (carga_por_mes <= CAPACIDADE_MENSAL_H).all()
print("  ✅ OK")

# ── V4: intervalo entre MPs consecutivas (R4) ──────────────
print("\n--- V4: intervalo entre MPs consecutivas (R4) ---")
viol_r4 = 0
for eq, grupo in df.groupby('Equipamento'):
    grupo = grupo.sort_values('_mes_venc')
    meses_aloc = grupo['_mes_num'].tolist()
    periodos   = grupo['_period'].tolist()
    for k in range(len(meses_aloc) - 1):
        if meses_aloc[k+1] - meses_aloc[k] < periodos[k]:
            viol_r4 += 1
assert viol_r4 == 0, f"❌ {viol_r4} violações de intervalo!"
print(f"  ✅ 0 violações de intervalo entre MPs consecutivas")

 CRONOGRAMA OTIMIZADO — 2026
      Equipamento  Duração (h) Mês alocado Mês mínimo Mês máximo
Arco Cirúrgico U1            5         Jan        Jan        Jan
Arco Cirúrgico U2            5         Jan        Jan        Jan
  Hemodinâmica U1            8         Jan        Jan        Jan
  Hemodinâmica U2            8         Jan        Jan        Jan
        Mamógrafo            4         Jan        Jan        Jan
           Raio-X            4         Jan        Jan        Jan
      Ressonância           10         Jan        Jan        Jan
        Tomógrafo            8         Jan        Jan        Jan
     Ultrassom U1            3         Jan        Jan        Jan
     Ultrassom U2            3         Jan        Jan        Jan
     Ultrassom U3            3         Jan        Jan        Jan
     Ultrassom U4            3         Jan        Jan        Jan
     Ultrassom U5            3         Jan        Jan        Jan
     Ultrassom U6            3         Jan        Jan        

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 6 — Comparação de cenários (item 4 do feedback)
#
# O orientador pediu: "não basta demonstrar que as manutenções
# foram antecipadas; precisamos evidenciar quantitativamente
# que a solução proposta representa uma melhoria em relação a
# uma programação de referência".
#
# Criamos 3 cenários:
#   A) Baseline "tudo no vencimento"  → pior caso operacional
#   B) Baseline "tudo no início da janela" → ingênuo, concentra
#   C) Solução PLIM otimizada           → proposta do TCC
#
# Métricas comparadas (operacionalmente relevantes):
#   1. Carga máxima mensal (h)         → quanto o pico aperta
#   2. Desvio-padrão da carga (h)      → balanceamento
#   3. Nº de meses com sobrecarga      → risco operacional
#   4. Folga mínima mensal (h)         → margem para corretivas
# ─────────────────────────────────────────────────────────────

def avaliar_cenario(nome, alocacao):
    """
    alocacao: dict {id_ordem: mes_alocado}
    Retorna dict com métricas do cenário.
    """
    carga = {m: 0 for m in MESES}
    for o in TODAS_AS_ORDENS:
        carga[alocacao[o['id']]] += o['duracao_h']
    cargas = np.array([carga[m] for m in MESES])
    return {
        'cenario':              nome,
        'carga_max':            cargas.max(),
        'carga_min':            cargas.min(),
        'desvio_padrao':        cargas.std(),
        'meses_sobrecarga':     int((cargas > CAPACIDADE_MENSAL_H).sum()),
        'folga_minima':         (CAPACIDADE_MENSAL_H - cargas).min(),
        'carga_por_mes':        carga,
    }

# ── Cenário A: tudo no vencimento ──────────────────────────
aloc_A = {o['id']: o['mes_max'] for o in TODAS_AS_ORDENS}

# ── Cenário B: tudo no início da janela ────────────────────
aloc_B = {o['id']: o['mes_min'] for o in TODAS_AS_ORDENS}

# ── Cenário C: solução PLIM ────────────────────────────────
aloc_C = {}
for o in TODAS_AS_ORDENS:
    for m in MESES:
        if (o['id'], m) in x and pulp.value(x[(o['id'], m)]) > 0.5:
            aloc_C[o['id']] = m
            break

cenarios = [
    avaliar_cenario('A) Tudo no vencimento',   aloc_A),
    avaliar_cenario('B) Tudo no início janela', aloc_B),
    avaliar_cenario('C) PLIM otimizado (TCC)',  aloc_C),
]

# ── Tabela comparativa ─────────────────────────────────────
df_comp = pd.DataFrame([
    {
        'Cenário':           c['cenario'],
        'Carga máx (h)':     f"{c['carga_max']:.0f}",
        'Carga mín (h)':     f"{c['carga_min']:.0f}",
        'Desvio-padrão (h)': f"{c['desvio_padrao']:.1f}",
        'Meses s/ sobrec.':  c['meses_sobrecarga'],
        'Folga mínima (h)':  f"{c['folga_minima']:.0f}",
    }
    for c in cenarios
])

print("=" * 78)
print(" COMPARAÇÃO DE CENÁRIOS (item 4 do feedback)")
print("=" * 78)
print(df_comp.to_string(index=False))

# ── Ganho percentual C vs A ────────────────────────────────
A, C = cenarios[0], cenarios[2]
print()
print("─" * 78)
print(" GANHO DA SOLUÇÃO OTIMIZADA EM RELAÇÃO AO BASELINE 'TUDO NO VENCIMENTO'")
print("─" * 78)
red_pico = (A['carga_max'] - C['carga_max']) / A['carga_max'] * 100
red_dp   = (A['desvio_padrao'] - C['desvio_padrao']) / A['desvio_padrao'] * 100
print(f"  Redução do pico mensal:    {red_pico:+.1f}%")
print(f"  Redução do desvio-padrão:  {red_dp:+.1f}%")
print(f"  Meses com sobrecarga:      {A['meses_sobrecarga']} → {C['meses_sobrecarga']}")
print(f"  Folga mínima mensal:       {A['folga_minima']:.0f} h → {C['folga_minima']:.0f} h")

# ── Carga mês a mês por cenário ────────────────────────────
print()
print("─" * 78)
print(" CARGA MENSAL POR CENÁRIO (h)")
print("─" * 78)
df_mensal = pd.DataFrame({
    'Mês':          NOMES_MESES,
    'A) Vencimento':[cenarios[0]['carga_por_mes'][m] for m in MESES],
    'B) Início':    [cenarios[1]['carga_por_mes'][m] for m in MESES],
    'C) PLIM':      [cenarios[2]['carga_por_mes'][m] for m in MESES],
    'Capacidade':   [CAPACIDADE_MENSAL_H] * 12,
})
print(df_mensal.to_string(index=False))

 COMPARAÇÃO DE CENÁRIOS (item 4 do feedback)
                 Cenário Carga máx (h) Carga mín (h) Desvio-padrão (h)  Meses s/ sobrec. Folga mínima (h)
   A) Tudo no vencimento            70            70               0.0                 0              141
B) Tudo no início janela           140             0              28.6                 0               71
 C) PLIM otimizado (TCC)            70            70               0.0                 0              141

──────────────────────────────────────────────────────────────────────────────
 GANHO DA SOLUÇÃO OTIMIZADA EM RELAÇÃO AO BASELINE 'TUDO NO VENCIMENTO'
──────────────────────────────────────────────────────────────────────────────
  Redução do pico mensal:    +0.0%
  Redução do desvio-padrão:  +nan%
  Meses com sobrecarga:      0 → 0
  Folga mínima mensal:       141 h → 141 h

──────────────────────────────────────────────────────────────────────────────
 CARGA MENSAL POR CENÁRIO (h)
──────────────────────────────────────────

/tmp/ipykernel_345/1279623241.py:85: RuntimeWarning: invalid value encountered in scalar divide
  red_dp   = (A['desvio_padrao'] - C['desvio_padrao']) / A['desvio_padrao'] * 100


In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 7 — Resumo final consolidado para o TCC
# ─────────────────────────────────────────────────────────────

print("=" * 78)
print(" RESUMO FINAL — MODELO PLIM PARA PROGRAMAÇÃO DE MPs (2026)")
print("=" * 78)
print(f"  Ordens programadas:          {len(df)}")
print(f"  Carga total de MP:           {carga_total} h")
print(f"  Capacidade mensal efetiva:   {CAPACIDADE_MENSAL_H} h")
print(f"  Capacidade anual:            {CAPACIDADE_MENSAL_H*12} h")
print(f"  Utilização média:            {carga_total/(CAPACIDADE_MENSAL_H*12):.1%}")
print(f"  Status do solver:            {pulp.LpStatus[status]}")
print(f"  Carga máxima mensal (PLIM):  {cenarios[2]['carga_max']:.0f} h")
print(f"  Carga mínima mensal (PLIM):  {cenarios[2]['carga_min']:.0f} h")
print(f"  Desvio-padrão da carga:      {cenarios[2]['desvio_padrao']:.1f} h")
print()
print("  Pontos atendidos do feedback do orientador:")
print("   [✓] FO fundamentada em balanceamento + sobrecarga + antecipação")
print("   [✓] Capacidade operacional calibrada (40% do tempo útil)")
print("   [✓] Restrição R4: intervalo entre MPs consecutivas do mesmo equip.")
print("   [✓] Comparação com 3 cenários de referência")
print("   [✓] Origem dos dados documentada (FONTE_DADOS)")
print("   [✓] Modelo permanece PLIM puro (sem complexidade extra)")
print

 RESUMO FINAL — MODELO PLIM PARA PROGRAMAÇÃO DE MPs (2026)
  Ordens programadas:          168
  Carga total de MP:           840 h
  Capacidade mensal efetiva:   211 h
  Capacidade anual:            2532 h
  Utilização média:            33.2%
  Status do solver:            Optimal
  Carga máxima mensal (PLIM):  70 h
  Carga mínima mensal (PLIM):  70 h
  Desvio-padrão da carga:      0.0 h

  Pontos atendidos do feedback do orientador:
   [✓] FO fundamentada em balanceamento + sobrecarga + antecipação
   [✓] Capacidade operacional calibrada (40% do tempo útil)
   [✓] Restrição R4: intervalo entre MPs consecutivas do mesmo equip.
   [✓] Comparação com 3 cenários de referência
   [✓] Origem dos dados documentada (FONTE_DADOS)
   [✓] Modelo permanece PLIM puro (sem complexidade extra)


<function print(*args, sep=' ', end='\n', file=None, flush=False)>